# APR18 LAOS：DWT 多分辨率分析与独立复核

这是**可执行的复现实验 Notebook**。使用原始 `APR18 SS SAOS STEPS LAOS UDLAOS.xlsx` 的 `LAOS` 工作表，不使用先前保存的图像或 CSV 作为计算输入。

方法：原始时间戳拟合 1、3、5 次谐波；从 t=7 s 开始取五个完整周期；将 Γ=10 的应力插值至 1,536 个等距点；使用**NumPy 实现的 db4 正交周期化 DWT**；提取 A8、D1–D8、D6 与三次谐波相关性；以独立 FFT 和重构误差作为检查。

所需 Python 包：`numpy`、`matplotlib`、`artifact_tool`。把原始 Excel 与此 Notebook 放在同一目录。注意标准 PyWavelets 的 symmetric 边界模式与此处的 periodization 不完全相同。

In [1]:
#!/usr/bin/env python3
"""Reproducible APR18 human-blood LAOS db4-DWT verification demo.

Input : APR18 SS SAOS STEPS LAOS UDLAOS.xlsx (worksheet LAOS)
Outputs: summary.csv, components.csv, figures, verification.json

Dependencies: numpy, matplotlib, artifact_tool (Excel reader).
Wavelets: an explicit *periodized* orthogonal db4 filter bank, implemented
using NumPy (PyWavelets is not required). Phase at boundaries may differ from
PyWavelets' symmetric mode. No smoothing or denoising is applied.

Run: python APR18_LAOS_DWT_Verified.py --input "APR18 SS SAOS STEPS LAOS UDLAOS.xlsx" --out RESULTS_DIR
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
from pathlib import Path
from typing import Dict

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

FILENAME = 'APR18 SS SAOS STEPS LAOS UDLAOS.xlsx'
SHEET = 'LAOS'
OMEGA = 1.0  # rad/s
AMPLITUDES = (0.5, 1, 5, 10, 50, 100)
COLS_0BASED = (0, 4, 8, 12, 16, 20)  # time, stress, strain; fourth column blank
PERIODS = 5
START_S = 7.0
GRID_N = 1536  # contains exactly 5 complete cycles; endpoint excluded
LEVEL = 8

# Exact db4 (eight-tap) scaling filter in orthogonal analysis convention.
DB4_LO = np.array([
    0.2303778133088964, 0.7148465705529154,
    0.6308807679298587, -0.027983769416859854,
    -0.18703481171888114, 0.030841381835986965,
    0.0328830116668852, -0.010597401785069032,
], dtype=float)
DB4_HI = ((-1.0) ** np.arange(len(DB4_LO))) * DB4_LO[::-1]


def sha256(p: Path) -> str:
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_data(input_path: Path) -> Dict[float, np.ndarray]:
    """Read actual original workbook, keeping time, stress, strain % and Excel row."""
    from artifact_tool import Blob, SpreadsheetFile
    wb = SpreadsheetFile.import_xlsx(Blob.load(str(input_path)))
    sh = wb.worksheets.get_item(SHEET)
    # Explicitly read original source coordinates; no hidden caching or CSV dependency.
    matrix = sh.get_range('A1:W2000').values
    sets = {}
    for gamma, base in zip(AMPLITUDES, COLS_0BASED):
        header = str(matrix[0][base])
        expected = f'Gamma={gamma:g}'
        assert f'omega=1' in header and expected in header, (header, expected)
        sample = []
        for excel_row, row in enumerate(matrix, start=1):
            try:
                t, sigma, strain = (float(row[base]), float(row[base + 1]), float(row[base + 2]))
            except (TypeError, ValueError, IndexError):
                continue
            if np.isfinite([t, sigma, strain]).all():
                sample.append((t, sigma, strain, excel_row))
        arr = np.array(sample, dtype=float)
        assert arr.shape == (1996, 4), (gamma, arr.shape)
        assert np.all(np.diff(arr[:, 0]) > 0), f'Non-increasing times: gamma={gamma}'
        sets[gamma] = arr
    return sets


def fit_harmonics(t: np.ndarray, y: np.ndarray, orders=(1, 3, 5)):
    """Least-squares fit of DC, sin(n*omega*t), cos(n*omega*t)."""
    X = np.column_stack([np.ones_like(t)] +
                        [col for n in orders for col in (np.sin(n * OMEGA * t), np.cos(n * OMEGA * t))])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    amps = {n: float(math.hypot(b[1 + 2*j], b[2 + 2*j])) for j, n in enumerate(orders)}
    fitted = X @ b
    r2 = 1.0 - float(np.sum((y - fitted)**2) / np.sum((y - y.mean())**2))
    return b, amps, r2


def harmonic_component(t, b, n):
    j = (1, 3, 5).index(n)
    return b[1 + 2*j]*np.sin(n*OMEGA*t) + b[2 + 2*j]*np.cos(n*OMEGA*t)


def dwt_step(x: np.ndarray):
    """Orthogonal periodic db4 forward step; input length must be even."""
    if x.size % 2:
        raise ValueError('DWT length must be even')
    idx = (2*np.arange(x.size//2)[:,None] + np.arange(8)[None,:]) % x.size
    return np.sum(x[idx]*DB4_LO, axis=1), np.sum(x[idx]*DB4_HI, axis=1)


def idwt_step(a: np.ndarray, d: np.ndarray):
    """Inverse of dwt_step with periodic boundary condition."""
    if a.shape != d.shape:
        raise ValueError('Coefficient lengths differ')
    n = 2*a.size
    idx = (2*np.arange(n//2)[:,None] + np.arange(8)[None,:]) % n
    out = np.zeros(n, dtype=float)
    np.add.at(out, idx, a[:,None]*DB4_LO + d[:,None]*DB4_HI)
    return out


def decompose(y, level=LEVEL):
    if y.size % 2**level:
        raise ValueError('Signal length must be divisible by 2**level in this implementation')
    a = np.array(y, dtype=float, copy=True)
    ds = []
    for _ in range(level):
        a, d = dwt_step(a)
        ds.append(d)
    return a, ds


def reconstruct(a, ds):
    x = np.array(a, copy=True)
    for d in reversed(ds):
        x = idwt_step(x, d)
    return x


def reconstructed_band(y, band, level=LEVEL):
    """band=0 => A[level]; band in [1, level] => D[band]."""
    a, ds = decompose(y, level)
    zero_ds = [np.zeros_like(d) for d in ds]
    if band == 0:
        return reconstruct(a, zero_ds)
    if not 1 <= band <= level:
        raise ValueError('Invalid detail level')
    zero_ds[band-1] = ds[band-1]
    return reconstruct(np.zeros_like(a), zero_ds)


def corr(a, b):
    return float(np.corrcoef(a, b)[0,1])


def make_figures(outdir, grid, stress, fund, third, residual, d1, d6res, d6third, summary):
    mask = grid < (START_S + 3 * (2*math.pi))
    f, axs = plt.subplots(4,1,figsize=(11,10.5),sharex=True,layout='constrained')
    axs[0].plot(grid[mask],stress[mask],lw=1.2,label='Measured stress (interpolated)')
    axs[0].plot(grid[mask],fund[mask],lw=1.0,label='DC + fitted fundamental')
    axs[0].set_ylabel('Stress (Pa)'); axs[0].legend(loc='upper right')
    axs[0].set_title('(a) APR18 LAOS: original stress and harmonic baseline')
    axs[1].plot(grid[mask],residual[mask],lw=1,label='Measured - DC - fundamental')
    axs[1].plot(grid[mask],third[mask],lw=1,label='Fitted third harmonic')
    axs[1].set_ylabel('Residual (Pa)'); axs[1].legend(loc='upper right')
    axs[1].set_title('(b) Non-fundamental periodic components')
    axs[2].plot(grid[mask],d6res[mask],lw=1,label='D6 of residual')
    axs[2].plot(grid[mask],d6third[mask],lw=1,label='D6 of fitted 3rd harmonic')
    axs[2].set_ylabel('D6 detail (Pa)'); axs[2].legend(loc='upper right')
    axs[2].set_title('(c) db4 D6 band, r = %.4f (nominal 0.382 - 0.764 Hz)' % summary['d6_corr'])
    axs[3].plot(grid[mask],d1[mask],lw=0.65)
    axs[3].set_ylabel('D1 detail (Pa)');axs[3].set_xlabel('Time (s)')
    axs[3].set_title('(d) D1 captures fastest fluctuations, not uniquely material nonlinearity')
    for ax in axs: ax.grid(alpha=0.25)
    f.suptitle('VERIFIED APR18 LAOS db4-DWT demo | omega=1 rad/s, Gamma=10',fontsize=15)
    f.savefig(outdir/'APR18_LAOS_DWT_Verified.png',dpi=185)
    plt.close(f)

    fig,axs=plt.subplots(1,2,figsize=(11.2,4.2),layout='constrained')
    gammas = [row['Gamma'] for row in summary['amplitude_table']]
    rat3=[row['I3_I1_pct'] for row in summary['amplitude_table']]
    rat5=[row['I5_I1_pct'] for row in summary['amplitude_table']]
    x=np.arange(len(gammas))
    axs[0].plot(x,rat3,'o-',label='I3 / I1')
    axs[0].plot(x,rat5,'s-',label='I5 / I1')
    axs[0].set_xticks(x,[f'{v:g}' for v in gammas]);axs[0].set_xlabel('Gamma')
    axs[0].set_ylabel('Harmonic amplitude ratio (%)')
    axs[0].set_title('Harmonic regression on original timestamps');axs[0].legend()
    nvals=[1536,2048]
    axs[1].plot(nvals,[summary['dwt_sensitivity'][str(n)]['D6_corr'] for n in nvals], 'o-',label='D6 vs third')
    axs[1].plot(nvals,[summary['dwt_sensitivity'][str(n)]['D7_corr'] for n in nvals], 's-',label='D7 vs third')
    axs[1].set_xticks(nvals);axs[1].set_ylim(0.8,1.01)
    axs[1].set_xlabel('Interpolated samples in same 5-cycle window')
    axs[1].set_ylabel('Correlation (r)')
    axs[1].set_title('DWT band index depends on sample rate');axs[1].legend()
    for ax in axs: ax.grid(alpha=0.25)
    fig.savefig(outdir/'APR18_LAOS_Verification_Comparisons.png',dpi=185)
    plt.close(fig)


def run(input_path: Path, outdir: Path):
    outdir.mkdir(parents=True,exist_ok=True)
    sets=load_data(input_path)
    high=START_S+PERIODS*(2*math.pi/OMEGA)
    amp_rows=[]
    for gamma,arr in sets.items():
        t,y,strain,_=arr.T
        dt=np.diff(t)
        choose=(t>=START_S)&(t<high)
        b,amp,r2=fit_harmonics(t[choose],y[choose])
        _,str_amp,_=fit_harmonics(t[choose],strain[choose])
        amp_rows.append({
            'Gamma':gamma,'n_full':len(t),'n_window':int(np.sum(choose)),
            'dt_median_s':float(np.median(dt)),
            'dt_cv_pct':float(100*np.std(dt)/np.mean(dt)),
            'I1_Pa':amp[1], 'I3_Pa':amp[3], 'I5_Pa':amp[5],
            'I3_I1_pct':100*amp[3]/amp[1], 'I5_I1_pct':100*amp[5]/amp[1],
            'strain_1_pct':str_amp[1], 'strain_I3_I1_pct':100*str_amp[3]/str_amp[1],
            'harmonic_R2':r2,
        })
    with (outdir/'APR18_LAOS_Harmonic_Verified.csv').open('w',newline='',encoding='utf-8-sig') as f:
        wr=csv.DictWriter(f,fieldnames=list(amp_rows[0]));wr.writeheader();wr.writerows(amp_rows)

    t,sigma,strain,_=sets[10].T
    grid=np.linspace(START_S,high,GRID_N,endpoint=False)
    sampled=np.interp(grid,t,sigma)  # Linear interpolation; no smoothing
    b,amp,_=fit_harmonics(grid,sampled)
    fund=b[0]+harmonic_component(grid,b,1)
    third=harmonic_component(grid,b,3)
    residual=sampled-fund
    bands=[reconstructed_band(sampled,k) for k in range(1,LEVEL+1)]
    low=reconstructed_band(sampled,0)
    d6res=reconstructed_band(residual,6)
    d6third=reconstructed_band(third,6)
    recon_error=float(np.max(np.abs(low+np.sum(bands,axis=0)-sampled)))
    assert recon_error < 1e-9, recon_error
    assert abs(amp_rows[3]['I3_I1_pct']-9.79112727)<0.005
    assert abs(amp_rows[3]['I1_Pa']-0.18633876)<1e-5
    assert abs(amp_rows[3]['I3_Pa']-0.01824467)<1e-5
    d6_corr=corr(d6res,d6third)
    assert 0.95<d6_corr<0.99, d6_corr

    fft=np.fft.rfft(sampled-sampled.mean())
    fft_i1=2*abs(fft[PERIODS])/GRID_N
    fft_i3=2*abs(fft[3*PERIODS])/GRID_N
    sensitivity={}
    for n in (1536,2048):
        gi=np.linspace(START_S,high,n,endpoint=False)
        yi=np.interp(gi,t,sigma)
        bb,aa,_=fit_harmonics(gi,yi)
        f0=bb[0]+harmonic_component(gi,bb,1)
        c3=harmonic_component(gi,bb,3)
        fre=n/(PERIODS*2*math.pi/OMEGA)
        sensitivity[str(n)]={
            'sample_rate_Hz':float(fre),
            'I3_I1_pct':float(100*aa[3]/aa[1]),
            'D6_corr':corr(reconstructed_band(yi-f0,6),reconstructed_band(c3,6)),
            'D7_corr':corr(reconstructed_band(yi-f0,7),reconstructed_band(c3,7)),
        }
    cycle_ratios=[]
    for k in range(PERIODS):
        st=START_S+k*2*math.pi
        sel=(t>=st)&(t<st+2*math.pi)
        _,a,_=fit_harmonics(t[sel],sigma[sel])
        cycle_ratios.append(float(100*a[3]/a[1]))

    measures={
        'source': {'file':input_path.name,'sha256':sha256(input_path),'sheet':SHEET,
                   'gamma10_raw_range':'M5:O2000','n_per_amplitude':1996},
        'method': {'omega_rad_s':OMEGA,'start_s':START_S,'end_s_exclusive':high,
                   'cycles':PERIODS,'interpolation':'linear','grid_n':GRID_N,
                   'grid_endpoint_included':False,'wavelet':'db4','boundary_mode':'periodization',
                   'level':LEVEL,'nominal_freq_note':'approximate idealized dyadic subbands only'},
        'amplitude_table':amp_rows,
        'gamma10_grid':{'fs_Hz':GRID_N/(high-START_S),'I1_Pa':amp[1],
                        'I3_Pa':amp[3],'I5_Pa':amp[5],
                        'I3_I1_pct':100*amp[3]/amp[1]},
        'd6_corr':d6_corr,'d6_sd_Pa':float(np.std(d6res)),'d1_sd_Pa':float(np.std(bands[0])),
        'reconstruction_max_abs_error_Pa':recon_error,
        'fft_i1_Pa':float(fft_i1),'fft_i3_Pa':float(fft_i3),
        'fft_i3_i1_pct':float(100*fft_i3/fft_i1),
        'cyclewise_gamma10_i3_i1_pct':cycle_ratios,
        'dwt_sensitivity':sensitivity,
    }
    with (outdir/'APR18_LAOS_Verification.json').open('w',encoding='utf-8') as f:
        json.dump(measures,f,ensure_ascii=False,indent=2)

    keys=['time_s','stress_interp_Pa','DC_plus_fundamental_Pa','third_harmonic_Pa',
          'residual_minus_fundamental_Pa','A8_reconstructed_Pa']+[f'D{k}_reconstructed_Pa' for k in range(1,9)] + [
          'D6_of_residual_Pa','D6_of_third_harmonic_Pa']
    with (outdir/'APR18_LAOS_DWT_Components_Verified.csv').open('w',newline='',encoding='utf-8-sig') as f:
        wr=csv.writer(f);wr.writerow(keys)
        for i in range(GRID_N):
            wr.writerow([grid[i],sampled[i],fund[i],third[i],residual[i],low[i]]+
                        [vv[i] for vv in bands]+[d6res[i],d6third[i]])
    make_figures(outdir,grid,sampled,fund,third,residual,bands[0],d6res,d6third,measures)
    print('Original source SHA256:', measures['source']['sha256'])
    print('Gamma10 (original irregular timestamps): I1=%.9f Pa, I3=%.9f Pa, I3/I1=%.6f%%' %
          (amp_rows[3]['I1_Pa'],amp_rows[3]['I3_Pa'],amp_rows[3]['I3_I1_pct']))
    print('Gamma10 1536-grid D6 correlation: %.9f; reconstruction error %.3e Pa' %
          (d6_corr, recon_error))
    print('FFT gamma10 I3/I1 %.6f%%, D1 SD %.8f Pa, D6 SD %.8f Pa' %
          (measures['fft_i3_i1_pct'],measures['d1_sd_Pa'],measures['d6_sd_Pa']))
    print('Verified results written to:',outdir)
    return measures



## 运行：读取原始 Excel，生成验证文件和图像

下一个单元格将生成所有结果，包含 `APR18_LAOS_Verification.json`、谐波比率 CSV、DWT 分量 CSV 和两张图。

In [2]:
from pathlib import Path
input_path = Path("APR18 SS SAOS STEPS LAOS UDLAOS.xlsx")
outdir = Path("APR18_LAOS_verified_results")
results = run(input_path, outdir)

Original source SHA256: 142f36127c216ac705822ea21118411de4b57c5519cd1b047ca45ce0fde76837
Gamma10 (original irregular timestamps): I1=0.186338763 Pa, I3=0.018244665 Pa, I3/I1=9.791127%
Gamma10 1536-grid D6 correlation: 0.968773348; reconstruction error 5.753e-13 Pa
FFT gamma10 I3/I1 9.789026%, D1 SD 0.00163670 Pa, D6 SD 0.01201814 Pa
Verified results written to: APR18_LAOS_verified_results


In [3]:
print("Gamma=10 original-sample 3rd/1st harmonic ratio (%):", results["amplitude_table"][3]["I3_I1_pct"])
print("D6 corr:", results["d6_corr"])
print("Reconstruction error:", results["reconstruction_max_abs_error_Pa"])
print("DWT sensitivity:", results["dwt_sensitivity"])

Gamma=10 original-sample 3rd/1st harmonic ratio (%): 9.791127270575066
D6 corr: 0.9687733477167334
Reconstruction error: 5.752620602095249e-13
DWT sensitivity: {'1536': {'sample_rate_Hz': 48.89239851783025, 'I3_I1_pct': 9.78902640100362, 'D6_corr': 0.9687733477167334, 'D7_corr': 0.991872906206072}, '2048': {'sample_rate_Hz': 65.18986469044033, 'I3_I1_pct': 9.788099875203121, 'D6_corr': 0.8814240704858708, 'D7_corr': 0.9991590836114492}}


## 数据与解释约束

- CSV 中 `stress_interp_Pa` 是**线性插值值**，不能视为仪器新增的真实测量点。原始 Excel 中每个 Γ 含 1,996 个测量点。
- 三次谐波出现在 D6 的近似频带，但 D6 **不专属于三次谐波**；分解层级随采样率而改变。
- D1 的高频小波系数不能直接判定为仪器噪声。
- 本实验不能单独证明血液 rouleaux 的微观结构演化，亦未验证时间依赖的 transient event 检测能力。